# AML File Consumer (HDFS file-based streaming)

Variante do consumer que usa **file-based streaming** sobre uma pasta HDFS em vez de Kafka.
Permite comparação de desempenho directa contra `aml_kafka_consumer.ipynb` (mesmo modelo, mesmas features, fonte diferente).

**Fluxo:** `aml_file_producer.ipynb` deposita CSVs em `stream/input/` → este consumer lê em append, aplica o modelo e grava em `stream/predictions_file/`.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

HDFS_BASE    = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
INPUT_PATH   = f"{HDFS_BASE}/stream/input"
MODEL_PATH   = f"{HDFS_BASE}/model/rf_aml_pipeline"
PRED_PATH    = f"{HDFS_BASE}/stream/predictions_file"
METRICS_PATH = f"{HDFS_BASE}/stream/metrics_file"
CHKPT_PRED   = f"{HDFS_BASE}/stream/chkpt_file_predictions"
CHKPT_METR   = f"{HDFS_BASE}/stream/chkpt_file_metrics"

spark = SparkSession.builder \
    .appName("AML File Consumer") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [ ]:
aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("From Account", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("To Account", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

model = PipelineModel.load(MODEL_PATH)
print("Modelo carregado.")

In [ ]:
raw = spark.readStream \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(INPUT_PATH)

# Anota timestamp de ingestão para medir latência no benchmark
parsed = raw.withColumn("ingest_ts", F.current_timestamp())

In [ ]:
def add_features(d):
    return (
        d.withColumn("log_amt_paid", F.log1p(F.col("Amount Paid")))
         .withColumn("log_amt_recv", F.log1p(F.col("Amount Received")))
         .withColumn("amt_diff", F.abs(F.col("Amount Paid") - F.col("Amount Received")))
         .withColumn("same_bank", (F.col("From Bank") == F.col("To Bank")).cast("int"))
         .withColumn("ccy_mismatch", (F.col("Receiving Currency") != F.col("Payment Currency")).cast("int"))
         .withColumn("hour", F.hour("Timestamp"))
    )

featured = add_features(parsed).na.fill({"hour": 0})

scored = (
    model.transform(featured)
        .withColumn("prob_fraud", vector_to_array("probability")[1])
        .withColumn(
            "status",
            F.when(F.col("prediction") == 1, F.lit("BLOQUEADA - suspeita de fraude"))
             .otherwise(F.lit("OK - fidedigna")),
        )
        .select(
            "ingest_ts", "Timestamp", "From Bank", "To Bank",
            "From Account", "To Account",
            "Amount Paid", "Receiving Currency", "Payment Format",
            F.col("Is Laundering").alias("truth"),
            F.col("prediction").cast("int").alias("prediction"),
            "prob_fraud", "status",
        )
)

In [ ]:
pred_query = scored \
    .withColumn("ingest_hour", F.date_format("ingest_ts", "yyyy-MM-dd-HH")) \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .partitionBy("ingest_hour") \
    .option("path", PRED_PATH) \
    .option("checkpointLocation", CHKPT_PRED) \
    .trigger(processingTime="5 seconds") \
    .start()

metrics = scored \
    .withWatermark("ingest_ts", "1 minute") \
    .groupBy(F.window("ingest_ts", "30 seconds")) \
    .agg(
        F.count("*").alias("n_msgs"),
        F.sum("prediction").alias("n_pred_pos"),
        F.sum("truth").alias("n_truth_pos"),
    )

metr_query = metrics.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", METRICS_PATH) \
    .option("checkpointLocation", CHKPT_METR) \
    .trigger(processingTime="10 seconds") \
    .start()

console = scored.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 20) \
    .trigger(processingTime="10 seconds") \
    .start()

spark.streams.awaitAnyTermination()

In [ ]:
for q in spark.streams.active:
    print("A parar:", q.name or q.id)
    q.stop()